# Constructing a Multi-Modal Document & Speech Pipeline -- Starter

You will configure a simulated Phi-4 Multimodal pipeline (text,
image, and audio processed together in one call), implement a
parallel Whisper baseline with explicit 16kHz resampling, and
package both as an Azure ML SDK v2 job -- all runnable with zero
GPU, zero live model downloads, and zero external files.


## Before you begin: environment setup


In [ ]:
%pip install soundfile datasets librosa azure-ai-ml pillow numpy


## Setup: Sample files

Provided for you -- builds the sample image and audio files this
activity uses.


In [ ]:
from PIL import Image, ImageDraw
import numpy as np
import soundfile as sf

img = Image.new("RGB", (600, 400), color=(245, 245, 245))
draw = ImageDraw.Draw(img)
draw.rectangle([40, 40, 560, 360], outline=(0, 0, 0), width=3)
draw.text((60, 60), "SAMPLE DOCUMENT", fill=(0, 0, 0))
draw.text((60, 100), "Invoice #10432", fill=(0, 0, 0))
draw.text((60, 130), "Total Due: $482.00", fill=(0, 0, 0))
img.save("sample_document.jpg")

native_rate = 44100  # deliberately NOT 16kHz, so resampling is genuinely needed
t = np.linspace(0, 2, native_rate * 2)
audio_data = (0.3 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)
sf.write("voice_memo.wav", audio_data, native_rate)

print("SETUP: sample_document.jpg and voice_memo.wav created")


## Task 1: Configure the Unified Phi-4 Multimodal Pipeline (10 min)

Complete a function that processes text, image, and audio together
in ONE call, using the real `<|image_1|>`/`<|audio_1|>` placeholder
token pattern Phi-4 Multimodal uses, and simulates its
Mixture-of-LoRAs modality routing conceptually.


In [ ]:
def route_modality_adapters(has_image, has_audio):
    # TODO: return a list of adapter names that would activate --
    # "vision_lora" if has_image, "audio_lora" if has_audio
    pass

def simulate_phi4_multimodal(image_path, audio_path, prompt):
    # TODO: load the image and audio file
    # TODO: call route_modality_adapters() to get the active adapters
    # TODO: build a simulated description string and a simulated
    # transcript string, and combine them into one response
    # TODO: return the response and the active adapters list
    pass

prompt = (
    "<|image_1|>\n<|audio_1|>\n"
    "Describe what the document shows and transcribe the audio note."
)
# TODO: call simulate_phi4_multimodal() and print both the response
# and the active adapters


Success Checklist:
- [ ] The prompt includes both `<|image_1|>` and `<|audio_1|>` placeholder tokens
- [ ] `route_modality_adapters` correctly returns adapter names based on which modalities are present
- [ ] The function returns ONE combined response covering both modalities, not two separate outputs


## Task 2: Whisper Baseline with 16kHz Resampling + Azure ML Job (10 min)

Implement a parallel Whisper-style transcription function with
explicit resampling, then package both pipelines as an Azure ML
SDK v2 job.


In [ ]:
import librosa

def simulate_whisper_transcribe(audio_path):
    # TODO: read the audio file
    # TODO: check if the sample rate is 16000; if not, resample with
    # librosa.resample
    # TODO: return a simulated transcript string mentioning the
    # resampled duration and target rate
    pass

# TODO: call simulate_whisper_transcribe() and print the result

# --- Package both pipelines as an Azure ML SDK v2 job ---
import os
os.makedirs("./pipeline_script", exist_ok=True)

# TODO: write a pipeline.py file to ./pipeline_script that prints
# messages simulating both the Phi-4 and Whisper pipelines running

from azure.ai.ml import command
from azure.ai.ml.entities import Environment

# TODO: build an Environment and a command() job referencing your
# pipeline script, then print its environment name and compute target

# TODO: run pipeline_script/pipeline.py locally via subprocess to
# confirm its logic works before any real submission


Success Checklist:
- [ ] `simulate_whisper_transcribe` checks the sample rate and resamples with `librosa.resample` when needed
- [ ] The Azure ML `command()` job references a real `pipeline.py` file and a named compute target
- [ ] Running `pipeline.py` locally completes without error


## Task 3: Multimodal Architecture Playbook (5 min)

Write a short playbook explaining: why resampling audio is a strict
requirement for models like Whisper; how multimodal networks use
specialized linear projectors to map visual patches into a shared
transformer sequence space; and the cost/latency savings of a
single-backbone native architecture versus a legacy multi-model
pipeline. (250-400 words)


*(Write your 250-400 word playbook here, addressing all three
required points explicitly.)*


## Key Points

- Phi-4 Multimodal processes all three modalities in ONE call;
  Whisper is a specialized, audio-only pipeline requiring its own
  explicit preprocessing step.
- Resampling is a silent-failure risk, not a crash risk -- always
  check and resample before passing audio to either model.
- Job packaging logic (Environment, command()) can be fully built
  and tested locally before any real Azure submission.
